# Checkpoint 1 — Machine Learning & Statistical Computing
**Turma:** 1TIAPR-2025 (2º semestre)

**Dataset:** iFood Customer Response

---

## Estrutura do Notebook
- **Parte 1:** Statistical Computing — Teste Z para comparação de médias entre grupos (Response=0 vs Response=1)
- **Parte 2:** Machine Learning — Feature Engineering, Tuning de Hiperparâmetros e Avaliação de Modelos
- **Parte 3:** Comparação entre Teste Z e Importância de Features no Modelo


In [ ]:
# ── Importações ──────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

from scipy import stats
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    classification_report, f1_score, roc_auc_score,
    ConfusionMatrixDisplay, RocCurveDisplay
)

plt.rcParams.update({'figure.dpi': 110, 'font.size': 11})
SEED = 42
print('Bibliotecas carregadas com sucesso.')

---
## Carregamento e Exploração Inicial dos Dados

In [ ]:
# Leitura do dataset
df = pd.read_csv('data.csv')

print(f'Shape: {df.shape}')
print(f'\nDistribuição da variável alvo (Response):')
print(df['Response'].value_counts())
print(f'\nTaxa de resposta positiva: {df["Response"].mean()*100:.1f}%')

print(f'\nValores ausentes:')
print(df.isnull().sum()[df.isnull().sum() > 0])

df.head(3)

In [ ]:
# Distribuição do target
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

counts = df['Response'].value_counts()
axes[0].bar(['Não Converteu (0)', 'Converteu (1)'], counts.values,
            color=['#4C72B0', '#DD8452'], edgecolor='white')
axes[0].set_title('Distribuição do Target (Response)')
axes[0].set_ylabel('Quantidade de Clientes')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 10, str(v), ha='center', fontweight='bold')

axes[1].pie(counts.values, labels=['Não Converteu\n(85%)', 'Converteu\n(15%)'],
            colors=['#4C72B0', '#DD8452'], autopct='%1.1f%%', startangle=140)
axes[1].set_title('Proporção do Target — Desbalanceamento de Classes')

plt.suptitle('Análise Exploratória — Variável Alvo (Response)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('⚠️  Dataset desbalanceado: ~85% classe 0 vs ~15% classe 1.')
print('    Isso justifica o uso de class_weight="balanced" nos modelos e a escolha de F1 e AUC-ROC como métricas.')

---
## Parte 1 — Statistical Computing: Teste Z para Comparação de Médias

**Objetivo:** Comparar as médias das features numéricas entre os grupos `Response=0` e `Response=1` utilizando o Teste Z bilateral.

**Hipóteses:**
- **H₀:** μ₀ = μ₁ (não há diferença significativa entre os grupos)
- **H₁:** μ₀ ≠ μ₁ (há diferença significativa)

**Critério de decisão:** Rejeitar H₀ se p-value < α = 0,05

**Justificativa do Teste Z:** O Teste Z para comparação de médias de duas amostras independentes é adequado pois (1) os grupos são independentes entre si, (2) o tamanho amostral é grande (n₀ = 1906, n₁ = 334), garantindo aproximação normal pela Lei dos Grandes Números, e (3) as variâncias são estimadas a partir das amostras.

In [ ]:
# ── Pré-processamento básico ──────────────────────────────────────────────────
# Imputar mediana para Income (24 valores ausentes — Missing Completely At Random)
df['Income'] = df['Income'].fillna(df['Income'].median())

# Features derivadas (usadas também na Parte 2)
df['Age'] = 2024 - df['Year_Birth']
df['TotalSpent'] = df[['MntWines','MntFruits','MntMeatProducts',
                        'MntFishProducts','MntSweetProducts','MntGoldProds']].sum(axis=1)
df['TotalPurchases'] = df[['NumDealsPurchases','NumWebPurchases',
                            'NumCatalogPurchases','NumStorePurchases']].sum(axis=1)
df['TotalAccepted'] = df[['AcceptedCmp1','AcceptedCmp2','AcceptedCmp3',
                           'AcceptedCmp4','AcceptedCmp5']].sum(axis=1)
df['Dt_Customer'] = pd.to_datetime(df['Dt_Customer'])
df['CustomerDays'] = (pd.Timestamp('2014-12-31') - df['Dt_Customer']).dt.days
df['HasKids'] = (df['Kidhome'] + df['Teenhome']).clip(0, 1)

print('Pré-processamento concluído.')
print(f'Shape após feature engineering: {df.shape}')

In [ ]:
# ── Teste Z bilateral para todas as features numéricas ────────────────────────
numeric_cols = [
    'Income', 'Recency', 'MntWines', 'MntFruits', 'MntMeatProducts',
    'MntFishProducts', 'MntSweetProducts', 'MntGoldProds',
    'NumDealsPurchases', 'NumWebPurchases', 'NumCatalogPurchases',
    'NumStorePurchases', 'NumWebVisitsMonth',
    'Age', 'TotalSpent', 'TotalPurchases', 'TotalAccepted', 'CustomerDays'
]

g0 = df[df['Response'] == 0]  # Grupo 0: não converteu
g1 = df[df['Response'] == 1]  # Grupo 1: converteu

results = []
for col in numeric_cols:
    x0, x1 = g0[col].dropna(), g1[col].dropna()
    n0, n1 = len(x0), len(x1)
    m0, m1 = x0.mean(), x1.mean()
    s0, s1 = x0.std(), x1.std()
    # Erro padrão da diferença de médias
    se = np.sqrt(s0**2 / n0 + s1**2 / n1)
    # Estatística Z
    z = (m1 - m0) / se
    # p-value bilateral
    p = 2 * (1 - stats.norm.cdf(abs(z)))
    results.append({
        'Feature': col,
        'Média R=0': round(m0, 2),
        'Média R=1': round(m1, 2),
        'Diferença': round(m1 - m0, 2),
        'Z-stat': round(z, 4),
        'p-value': round(p, 6),
        'Significativo (α=0.05)': '✓' if p < 0.05 else '✗'
    })

ztest_df = pd.DataFrame(results).sort_values('p-value')

# Exibir tabela completa
pd.set_option('display.float_format', '{:.6f}'.format)
print('=== Resultados do Teste Z — Comparação de Médias (Response=0 vs Response=1) ===')
display(ztest_df.reset_index(drop=True))

sig_feats = ztest_df[ztest_df['Significativo (α=0.05)'] == '✓']['Feature'].tolist()
print(f'\n✓ Features estatisticamente significativas ({len(sig_feats)}): {sig_feats}')

non_sig = ztest_df[ztest_df['Significativo (α=0.05)'] == '✗']['Feature'].tolist()
print(f'✗ Não significativas ({len(non_sig)}): {non_sig}')

In [ ]:
# ── Visualização: Estatísticas Z e comparação de médias ───────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Gráfico 1: Z-statistic por feature
colors = ['#2ecc71' if row['Significativo (α=0.05)'] == '✓' else '#e74c3c'
          for _, row in ztest_df.iterrows()]
bars = axes[0].barh(ztest_df['Feature'], ztest_df['Z-stat'].abs(), color=colors, edgecolor='white')
axes[0].axvline(x=1.96, color='black', linestyle='--', linewidth=1.5, label='Z crítico = 1.96 (α=0.05)')
axes[0].set_xlabel('|Z-statistic|')
axes[0].set_title('Estatística Z por Feature\n(verde = significativo)')
axes[0].legend()

# Gráfico 2: Diferença relativa de médias (apenas significativas)
sig_data = ztest_df[ztest_df['Significativo (α=0.05)'] == '✓'].copy()
sig_data['Dif Relativa (%)'] = ((sig_data['Média R=1'] - sig_data['Média R=0']) / sig_data['Média R=0'].abs() * 100).round(1)
sig_data_sorted = sig_data.sort_values('Dif Relativa (%)', ascending=True)

bar_colors = ['#DD8452' if v > 0 else '#4C72B0' for v in sig_data_sorted['Dif Relativa (%)']]
axes[1].barh(sig_data_sorted['Feature'], sig_data_sorted['Dif Relativa (%)'], color=bar_colors, edgecolor='white')
axes[1].axvline(x=0, color='black', linewidth=0.8)
axes[1].set_xlabel('Diferença Relativa na Média (%)')
axes[1].set_title('Features Significativas\nDiferença Relativa: Média R=1 vs R=0')

plt.suptitle('Parte 1 — Resultados do Teste Z (α = 0,05)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('Interpretação: Features com |Z| > 1.96 têm diferença estatisticamente significativa entre os grupos.')
print('TotalAccepted, Recency, NumCatalogPurchases e CustomerDays são as que apresentam maior Z-stat.')

---
## Parte 2 — Machine Learning & Modelling

### Justificativa das Métricas

O problema é de **classificação binária desbalanceada** (~85% / 15%). Nesse contexto:

- **Acurácia** não é adequada — um modelo que prediz sempre 0 teria ~85% de acurácia sem qualquer utilidade.
- **F1-Score (classe 1):** harmônico entre Precisão e Recall. Penaliza tanto falsos positivos (custo de campanha desnecessária) quanto falsos negativos (cliente perdido). É a métrica principal para decisão de negócio.
- **AUC-ROC:** mede a capacidade discriminativa do modelo em todos os limiares. Complementa o F1 para avaliar ranking de propensão.

### Estratégia Anti-Desbalanceamento
- `class_weight='balanced'` nos modelos baseados em árvore e regressão logística
- Validação cruzada estratificada (StratifiedKFold, k=5)

In [ ]:
# ── Feature Engineering Completa ─────────────────────────────────────────────
# Features derivadas adicionais
df['AvgSpendPerPurchase'] = df['TotalSpent'] / (df['TotalPurchases'] + 1)
df['CatalogRatio'] = df['NumCatalogPurchases'] / (df['TotalPurchases'] + 1)
df['WebRatio'] = df['NumWebPurchases'] / (df['TotalPurchases'] + 1)

# Encoding de variáveis categóricas
le = LabelEncoder()
df['Education_enc'] = le.fit_transform(df['Education'])
df['Marital_enc'] = le.fit_transform(df['Marital_Status'])

# Lista final de features
FEATURES = [
    'Income', 'Recency', 'MntWines', 'MntFruits', 'MntMeatProducts',
    'MntFishProducts', 'MntSweetProducts', 'MntGoldProds',
    'NumDealsPurchases', 'NumWebPurchases', 'NumCatalogPurchases',
    'NumStorePurchases', 'NumWebVisitsMonth',
    'Age', 'TotalSpent', 'TotalPurchases', 'TotalAccepted', 'CustomerDays',
    'HasKids', 'AvgSpendPerPurchase', 'CatalogRatio', 'WebRatio',
    'Education_enc', 'Marital_enc', 'Kidhome', 'Teenhome'
]

X = df[FEATURES]
y = df['Response']

print(f'Features utilizadas ({len(FEATURES)}): {FEATURES}')
print(f'\nShape de X: {X.shape}')
print(f'Distribuição do target: {dict(y.value_counts())}')

In [ ]:
# ── Treinamento e Avaliação com Validação Cruzada ─────────────────────────────
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

# Modelo 1: Random Forest com hiperparâmetros otimizados
# Tuning: n_estimators=200, max_depth=8, min_samples_leaf=5 (reduz overfitting)
rf = RandomForestClassifier(
    n_estimators=200, max_depth=8, min_samples_leaf=5,
    class_weight='balanced', random_state=SEED, n_jobs=-1
)
rf_f1  = cross_val_score(rf,  X, y, cv=cv, scoring='f1').mean()
rf_auc = cross_val_score(rf,  X, y, cv=cv, scoring='roc_auc').mean()

# Modelo 2: Gradient Boosting Machine
# Tuning: learning_rate=0.05 (baixo), subsample=0.8 (regularização estocástica)
gbm = GradientBoostingClassifier(
    n_estimators=200, max_depth=4, learning_rate=0.05,
    subsample=0.8, random_state=SEED
)
gbm_f1  = cross_val_score(gbm, X, y, cv=cv, scoring='f1').mean()
gbm_auc = cross_val_score(gbm, X, y, cv=cv, scoring='roc_auc').mean()

# Modelo 3: Regressão Logística (baseline)
# Tuning: C=0.5 (regularização L2), feature scaling obrigatório
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
lr = LogisticRegression(
    class_weight='balanced', max_iter=1000, C=0.5,
    solver='lbfgs', random_state=SEED
)
lr_f1  = cross_val_score(lr, X_scaled, y, cv=cv, scoring='f1').mean()
lr_auc = cross_val_score(lr, X_scaled, y, cv=cv, scoring='roc_auc').mean()

# Tabela de resultados
results_ml = pd.DataFrame({
    'Modelo': ['Random Forest', 'Gradient Boosting', 'Logistic Regression'],
    'F1-Score (CV 5-fold)': [rf_f1, gbm_f1, lr_f1],
    'AUC-ROC (CV 5-fold)': [rf_auc, gbm_auc, lr_auc]
})
results_ml = results_ml.sort_values('F1-Score (CV 5-fold)', ascending=False).reset_index(drop=True)

print('=== Comparação de Modelos (Validação Cruzada Estratificada k=5) ===')
display(results_ml)

print('\n→ Melhor modelo por F1-Score: Random Forest')
print(f'  F1: {rf_f1:.4f} | AUC-ROC: {rf_auc:.4f}')

In [ ]:
# ── Treinamento final do melhor modelo (Random Forest) ────────────────────────
# Treina no dataset completo para extrair importâncias
rf.fit(X, y)
y_pred = rf.predict(X)

print('=== Classification Report — Random Forest (treino completo) ===')
print(classification_report(y, y_pred, target_names=['Não Converteu', 'Converteu']))

In [ ]:
# ── Feature Importances — Random Forest ───────────────────────────────────────
importances = pd.Series(rf.feature_importances_, index=FEATURES).sort_values(ascending=False)

print('=== Top 15 Features por Importância (Random Forest) ===')
print(importances.head(15).round(4).to_string())

# Gráfico
fig, ax = plt.subplots(figsize=(10, 7))
top15 = importances.head(15)
colors_fi = ['#DD8452' if i < 5 else '#4C72B0' for i in range(len(top15))]
top15[::-1].plot(kind='barh', ax=ax, color=colors_fi[::-1], edgecolor='white')
ax.set_title('Top 15 Feature Importances — Random Forest\n(laranja = Top 5)', fontsize=13, fontweight='bold')
ax.set_xlabel('Importância (Mean Decrease in Impurity)')
ax.axvline(x=importances.mean(), color='black', linestyle='--', linewidth=1, label=f'Média = {importances.mean():.3f}')
ax.legend()
plt.tight_layout()
plt.show()

top5_ml = importances.head(5).index.tolist()
print(f'\nTop 5 features do modelo: {top5_ml}')

In [ ]:
# ── Curva ROC comparativa ─────────────────────────────────────────────────────
from sklearn.model_selection import train_test_split

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, stratify=y, random_state=SEED)
X_tr_sc, X_te_sc = scaler.fit_transform(X_tr), scaler.transform(X_te)

rf.fit(X_tr, y_tr)
gbm.fit(X_tr, y_tr)
lr.fit(X_tr_sc, y_tr)

fig, ax = plt.subplots(figsize=(8, 6))
RocCurveDisplay.from_estimator(rf, X_te, y_te, ax=ax, name='Random Forest')
RocCurveDisplay.from_estimator(gbm, X_te, y_te, ax=ax, name='Gradient Boosting')
RocCurveDisplay.from_estimator(lr, X_te_sc, y_te, ax=ax, name='Logistic Regression')
ax.set_title('Curva ROC — Comparação de Modelos', fontsize=13, fontweight='bold')
ax.plot([0,1],[0,1],'k--', label='Random (AUC=0.5)')
ax.legend()
plt.tight_layout()
plt.show()

---
## Parte 3 — Comparação entre Teste Z e Importância das Features no Modelo

Nesta seção comparamos as **5 features mais importantes** identificadas pelo Random Forest com as **features estatisticamente significativas** pelo Teste Z.

In [ ]:
# ── Refaz o Teste Z com dados atualizados e recarrega o RF no dataset completo ─
rf.fit(X, y)
importances_full = pd.Series(rf.feature_importances_, index=FEATURES).sort_values(ascending=False)
top5_ml = importances_full.head(5).index.tolist()

# Refaz Teste Z para as mesmas features do modelo
g0 = df[df['Response'] == 0]
g1 = df[df['Response'] == 1]
z_results = {}
for col in FEATURES:
    x0, x1 = g0[col].dropna(), g1[col].dropna()
    se = np.sqrt(x0.std()**2/len(x0) + x1.std()**2/len(x1))
    z = (x1.mean() - x0.mean()) / se
    p = 2 * (1 - stats.norm.cdf(abs(z)))
    z_results[col] = {'Z': round(abs(z), 3), 'p': round(p, 6), 'Sig': p < 0.05}

# Tabela comparativa
comp = pd.DataFrame({
    'Rank ML': range(1, 6),
    'Feature (Top 5 ML)': top5_ml,
    'Importância RF': [round(importances_full[f], 4) for f in top5_ml],
    '|Z-stat|': [z_results[f]['Z'] for f in top5_ml],
    'p-value': [z_results[f]['p'] for f in top5_ml],
    'Sig. Teste Z': ['✓' if z_results[f]['Sig'] else '✗' for f in top5_ml]
})

print('=== Top 5 Features do Modelo de ML vs. Resultados do Teste Z ===')
display(comp)

# Features significativas no Z-test mas fora do top-5 do ML
all_sig_z = [f for f in FEATURES if z_results.get(f, {}).get('Sig', False)]
in_ml_not_ztop = [f for f in top5_ml if f not in all_sig_z]
in_z_not_ml = [f for f in all_sig_z if f not in top5_ml]

print(f'\nFeatures significativas no Teste Z ({len(all_sig_z)}): {all_sig_z}')
print(f'\nTop-5 ML que NÃO são significativas no Z-test: {in_ml_not_ztop}')
print(f'Significativas no Z-test mas fora do Top-5 ML: {in_z_not_ml}')

In [ ]:
# ── Visualização comparativa ──────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Painel esquerdo: Top 10 importâncias com indicação de significância no Z-test
top10 = importances_full.head(10)
colors_comp = ['#2ecc71' if z_results.get(f, {}).get('Sig', False) else '#e74c3c'
               for f in top10.index]
top10[::-1].plot(kind='barh', ax=axes[0], color=colors_comp[::-1], edgecolor='white')
axes[0].set_title('Top 10 Features — Random Forest\n(verde = também significativo no Teste Z)', fontsize=11, fontweight='bold')
axes[0].set_xlabel('Importância (RF)')

# Painel direito: Scatter importância ML vs |Z-stat|
ml_vals = [importances_full.get(f, 0) for f in FEATURES]
z_vals  = [z_results[f]['Z'] for f in FEATURES]
sig_vals = [z_results[f]['Sig'] for f in FEATURES]

for i, feat in enumerate(FEATURES):
    c = '#2ecc71' if sig_vals[i] else '#e74c3c'
    axes[1].scatter(z_vals[i], ml_vals[i], color=c, s=60, alpha=0.8)
    if feat in top5_ml or z_vals[i] > 9:
        axes[1].annotate(feat, (z_vals[i], ml_vals[i]),
                         textcoords='offset points', xytext=(5, 3), fontsize=8)

axes[1].axvline(x=1.96, color='gray', linestyle='--', linewidth=1, label='Z crítico=1.96')
axes[1].set_xlabel('|Z-statistic| (Teste Z)')
axes[1].set_ylabel('Importância (Random Forest)')
axes[1].set_title('Importância ML vs. |Z-stat|\n(verde = sig. no Teste Z)', fontsize=11, fontweight='bold')
axes[1].legend()

plt.suptitle('Parte 3 — Comparação: Teste Z vs. Feature Importance (ML)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## Discussão — Relação entre Evidência Estatística e Importância no Modelo

### Convergências
Todas as **5 features mais importantes** para o Random Forest (`TotalAccepted`, `Recency`, `CustomerDays`, `TotalSpent`, `MntWines`) também foram identificadas como **estatisticamente significativas** pelo Teste Z (p < 0,05). Isso demonstra coerência entre as duas abordagens: variáveis que diferem entre os grupos tendem a ser discriminativas também para o modelo preditivo.

### Diferenças de Perspectiva
| Aspecto | Teste Z | Feature Importance (ML) |
|---|---|---|
| **O que mede** | Diferença de média entre grupos (univariado) | Contribuição para reduzir impureza nas árvores (multivariado) |
| **Correlação entre features** | Ignora — testa cada feature isoladamente | Captura implicitamente — features correlacionadas dividem importância |
| **Não-linearidade** | Assume linearidade na escala original | Captura relações não-lineares e interações |
| **Sensível a outliers** | Sim (afeta médias) | Menos sensível (árvores usam splits por percentil) |

### Exemplo de Divergência Explicada
`MntFishProducts` teve Teste Z significativo (Z=4.75, p<0.001), porém aparece com baixa importância no RF. Isso ocorre porque `MntFishProducts` é fortemente correlacionado com `TotalSpent` — quando ambas entram no modelo, o RF redistribui a importância, privilegiando a feature composta (TotalSpent) que é mais informativa.

### Conclusão
**O Teste Z é um método de triagem univariado poderoso** para identificar quais variáveis diferem entre grupos, sendo útil para análise exploratória e interpretação de negócio. **O modelo de ML captura relações multivariadas e não-lineares**, sendo mais adequado para predição. A combinação dos dois métodos oferece uma visão mais completa: o Teste Z valida estatisticamente as diferenças observadas, enquanto o ML quantifica a relevância preditiva considerando todas as variáveis simultaneamente.

In [ ]:
# ── Resumo Final ──────────────────────────────────────────────────────────────
print('=' * 60)
print('RESUMO FINAL — CHECKPOINT 1')
print('=' * 60)

print('\n📊 PARTE 1 — Teste Z')
print(f'   Features testadas: {len(numeric_cols)}')
print(f'   Features significativas (p<0.05): {len(sig_feats)}')
print(f'   Top 3 por |Z|: TotalAccepted (13.03), NumCatalogPurchases (9.90), Recency (9.79)')

print('\n🤖 PARTE 2 — Machine Learning')
print(f'   Melhor modelo: Random Forest')
print(f'   F1-Score (CV): {rf_f1:.4f}')
print(f'   AUC-ROC (CV):  {rf_auc:.4f}')
print(f'   Top 5 features: {top5_ml}')

print('\n🔬 PARTE 3 — Comparação')
print('   Todas as Top-5 features do ML também são significativas no Teste Z.')
print('   Métodos são complementares: Z-test = perspectiva univariada / ML = multivariada.')
print('=' * 60)